In [1]:
import os, shutil
import torch
import kagglehub
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

path = kagglehub.dataset_download("emmarex/plantdisease")
data_dir = os.path.join(path, "PlantVillage")

dup = os.path.join(data_dir, "PlantVillage")
if os.path.exists(dup):
    shutil.rmtree(dup)

mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness = 0.2, contrast = 0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean,std)
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

train_full = datasets.ImageFolder(data_dir, transform = train_transform)
test_full = datasets.ImageFolder(data_dir, transform = test_transform)

train_size = int(0.8 * len(train_full))
generator = torch.Generator().manual_seed(42)
indices = torch.randperm(len(train_full), generator = generator).tolist()

train_data = Subset(train_full, indices[:train_size])
test_data = Subset(test_full, indices[train_size:])

train_loader = DataLoader(train_data, batch_size = 32, shuffle = True)
test_loader = DataLoader(test_data, batch_size = 32)

classes = train_full.classes
print(len(train_data), len(test_data), len(classes))

c:\Users\bryan\Documents\Github\Repositories\Leaf-Disease-Detector\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


16510 4128 15


In [3]:
import torch.nn as nn
from torchvision import models

device = "cuda"

weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights = weights)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, len(classes))
model = model.to(device)

print(model.fc)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total:,} | trainable: {trainable:,}")

Linear(in_features=512, out_features=15, bias=True)
Total params: 11,184,207 | trainable: 7,695
